In [3]:
# Model
from pprint import pprint

from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://host.docker.internal:11434",
    model="qwen3.5:9b",
    temperature=0.48,
    num_predict=-1,  # unlimited output tokens — let the model write until it stops naturally
    num_thread=8,  # match available CPU cores
    repeat_penalty=1.0,  # disable repeat penalty — default (1.1) causes small models to stop early
    keep_alive="12m",  # keep model in VRAM for 12 minutes between runs
    think=True,  # enable Qwen3 chain-of-thought thinking — visible in response.thinking
    # num_ctx omitted — uses model default for qwen3.5:9b
)
pprint(llm)

ChatOllama(model='qwen3.5:9b', num_thread=8, num_predict=-1, repeat_penalty=1.0, temperature=0.48, keep_alive='12m', base_url='http://host.docker.internal:11434')


In [ ]:
# Run Model
from pprint import pprint

hello_world = llm.invoke("Hello, world!")
hello_world.pretty_print()
pprint(hello_world.model_dump(exclude={"content"}))

================================== Ai Message ==================================

Hello! 👋 How can I help you today?
{'additional_kwargs': {},
 'id': 'lc_run--019cbac2-13b6-79a0-a353-e76dd1a8e988-0',
 'invalid_tool_calls': [],
 'name': None,
 'response_metadata': {'created_at': '2026-03-04T21:30:41.432378Z',
                       'done': True,
                       'done_reason': 'stop',
                       'eval_count': 521,
                       'eval_duration': 31462950013,
                       'load_duration': 293478583,
                       'logprobs': None,
                       'model': 'qwen3.5:9b',
                       'model_name': 'qwen3.5:9b',
                       'model_provider': 'ollama',
                       'prompt_eval_count': 14,
                       'prompt_eval_duration': 334737792,
                       'total_duration': 34163969875},
 'tool_calls': [],
 'type': 'ai',
 'usage_metadata': {'input_tokens': 14,
                    'output_tokens': 

In [2]:
# Tools
import trafilatura
from langchain_community.utilities import SearxSearchWrapper
from langchain_core.tools import tool

searxng_wrapper = SearxSearchWrapper(searx_host="http://searxng:8080")


@tool
def searxng_search(query: str) -> str:
    """Search using SearxNG, a self-hosted metasearch engine aggregating
    results from various sources. Fetches and returns the full text content
    of the top result pages for comprehensive research.

    Args:
        query: The search terms to look up, phrased as a concise natural language query.
    """
    searxng_results = searxng_wrapper.results(query, num_results=12)
    trafilatura_results = []
    for i, result in enumerate(searxng_results, start=1):
        url = result.get("link", "")
        title = result.get("title", "").strip()
        text = trafilatura.extract(
            trafilatura.fetch_url(url),
            include_comments=False,
            include_tables=True,
            no_fallback=False,
        )
        heading = title if title else url
        trafilatura_results.append(
            f"## Search Result {i}: {heading}\n\n**Source:** [{url}]({url})\n\n{text}"
        )

    if not trafilatura_results:
        raise ValueError(f"No content could be extracted for query: {query}")

    return "\n\n---\n\n".join(trafilatura_results)


# tool_results = searxng_search.invoke("What is the meaning of the tarot card 'The Hermit'?")
# pprint(tool_results)

In [ ]:
from pathlib import Path

from langchain_core.messages import HumanMessage, ToolMessage

from src.practices import TAROT_CARDS


def ingest_article(practice):
    name = practice["name"]
    slug = practice["slug"]
    category = "Tarot"

    tool_message = ToolMessage(
        content=searxng_search.invoke(
            f'"{name}" {category.lower()} meaning significance themes '
            "symbolism representation interpretation correspondences spirituality"
        ),
        tool_call_id="searxng_search",
    )
    # tool_message.pretty_print()

    human_message = HumanMessage(
        content=f"""\
    Using the provided search results, write a comprehensive reference article \
    about **{name}** ({category}). Target length: 1200 words.

    # {name}

    Cover its archetypal meaning and spiritual themes, the key symbols and what \
    they reveal (use **bold** for symbol names), how its energy applies to \
    relationships, career, and inner development, its place in the broader \
    {category} category, and any notable traditional correspondences \
    (element, planet, zodiac, number). Write in flowing paragraphs without \
    sub-headings — let the ideas connect naturally rather than following a \
    fixed structure.

    Use only information from the search results. Do not include affirmations, \
    mantras, journal prompts, or reflection questions.
    """
    )

    final_message = llm.invoke([tool_message, human_message])
    final_message.pretty_print()

    output_directory = Path(f"../../output/{category.lower()}")
    output_directory.mkdir(parents=True, exist_ok=True)
    output_filename = f"{slug}.md"
    output_path = output_directory / output_filename
    output_path.write_text(final_message.content, encoding="utf-8")
    print(f"\nWritten to: {output_path.resolve()}\n")


for practice in TAROT_CARDS:
    ingest_article(practice)